# YouTube RAG Chatbot - Linux Version
# Runs locally on Parrot OS / Jupyter Notebook.


In [1]:
# Install dependencies from the project terminal:
# pip install -r requirements.txt


In [2]:
# #here we will try to implement using chain also learn the flow of program
# user asks a question
# question--->question--->question\
#                                  \
#                                   |-> prompt | llm | response
#                                  /
# question--->retriever--->context/

# Details:
# question is asked we will use that question in retriever to get the matching values / result to narrow down the context
# why?to narrow down the context as we dont know how long will the youtube transcript are and need to save work efficiently
# and use then usse context and question for prompt for llm and get answers

##**To run for any video change id see last 4 non commented cells**

In [3]:
import os
from youtube_transcript_api import YouTubeTranscriptApi


In [4]:
def fetch_transcript(video_id: str) -> str:
    ytt_api = YouTubeTranscriptApi()
    result = ytt_api.fetch(video_id)
    transcript = " "
    for snippets in result:
      transcript=transcript+" "+snippets.text
    return transcript

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_transcript(transcript: str):
    splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=10)
    chunks = splitter.create_documents([transcript])
    return chunks

In [6]:
from langchain_cohere import CohereEmbeddings

# Pass the API key string directly
embed_model = CohereEmbeddings(
    cohere_api_key=os.getenv("COHERE_API_KEY"), 
    model="embed-english-v3.0"
)

In [7]:
#store in vectorDB
from langchain_chroma import Chroma
def store_in_chroma(chunks):
    # This creates the store and saves the embeddings
    vectorStore = Chroma.from_documents(chunks, embed_model,collection_name="MonkeyDB")
    return vectorStore

In [8]:
# vectorStore.get()

In [9]:
# vectorStore.get()["documents"]

In [10]:
# doc=retriever.invoke("where does black hole is?")
# print(doc)

In [11]:
# #extract pagecontent from context from retrieve
# context=""
# for i in doc:
#     context=context+" "+i.page_content
# print(context)

In [12]:
from langchain_cohere import ChatCohere
from langchain_core.messages import AIMessage, HumanMessage
# Define the Cohere LLM
model=llm = ChatCohere(
    cohere_api_key=os.getenv("COHERE_API_KEY"), 
    model="command-a-03-2025"
)

In [13]:
from langchain_core.prompts import PromptTemplate
prompt=PromptTemplate(template="""You are a helpfull agent Kindly provide me answer to the question {question} strictly using the give context only {context} in english. If nothing is present or unable to answer say I dont know""",input_variables=["question","context"])
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are a helpfull agent Kindly provide me answer to the question {question} strictly using the give context only {context} in english. If nothing is present or unable to answer say I dont know')

In [14]:
# prompt_result=prompt.invoke({"question":"what is black hole and its size","context":context})
# prompt_result

In [15]:
# result=model.invoke(prompt_result)
# print(result)

In [16]:
# steps make parallel chain for context generation and question accusition

In [17]:
def formatContext(ip):
    context=""
    for i in ip:
        context=context+" "+i.page_content
    return context

In [18]:
# formatContext(doc)

In [19]:
# parallelChain.invoke("what is size of black hole")

In [20]:
from langchain_core.runnables import RunnableLambda
ingestion_chain = (
    RunnableLambda(fetch_transcript)    # Takes video_id string -> returns transcript string
    | RunnableLambda(split_transcript)  # Takes transcript string -> returns chunks
    | RunnableLambda(store_in_chroma)   # Takes chunks -> saves to DB -> returns vectorStore object
)

## **Here Change id of videoand run ingestion_chain  then run parallel chain with you query**

In [21]:
# Enter a YouTube URL in the next cell.
# The original hardcoded video ID has been removed.


In [22]:
from urllib.parse import urlparse, parse_qs

url = input("Enter YouTube URL: ").strip()

def get_video_id(url):
    parsed = urlparse(url)
    if parsed.hostname == "youtu.be":
        return parsed.path.lstrip("/").split("/")[0]
    if parsed.hostname in ("www.youtube.com", "youtube.com", "m.youtube.com"):
        if "v" in parse_qs(parsed.query):
            return parse_qs(parsed.query)["v"][0]
        if parsed.path.startswith("/shorts/"):
            return parsed.path.split("/shorts/")[1].split("/")[0]
    return url

video_id = get_video_id(url)
print("Video ID:", video_id)
vectorStore = ingestion_chain.invoke(video_id)


Enter YouTube URL:  https://www.youtube.com/watch?v=dQw4w9WgXcQ


In [23]:
retriever=vectorStore.as_retriever(search_kwargs={"k":5})

In [24]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser
parallelChain=RunnableParallel({
    "question":RunnablePassthrough(),
    "context":retriever|RunnableLambda(formatContext)
})

In [25]:
chain=parallelChain|prompt|model|StrOutputParser()
result=chain.invoke("what is this video about")#ask questions here
print(result)

UnprocessableEntityError: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': 'abc07660e91ceda4128d97c5b06d0353', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '20', 'x-trial-endpoint-call-remaining': '19', 'date': 'Thu, 30 Jul 2026 19:40:16 GMT', 'x-envoy-upstream-service-time': '134', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 422, body: {'error_type': 'NO_VALID_RESPONSE_GENERATED', 'id': '44d9ab12-c106-4ec4-9399-61fa1a8031db', 'message': 'No valid response generated. Try updating messages'}

## Linux Environment Demonstration
This project runs inside a Parrot OS virtual machine using VirtualBox.


In [ ]:
import platform, subprocess
print("Operating System:", platform.system())
print("Platform:", platform.platform())
print("Python Version:", platform.python_version())
print("Kernel:")
print(subprocess.check_output(["uname", "-a"]).decode().strip())
